# Лабораторная работа (оптимизация), **вариант 8** — задание **1.8**

**Цель (по методичке):** исследование условного экстремума; необходимые условия 1-го порядка (ККТ); необходимые/достаточные условия 2-го порядка; стационарные точки и значения функции.

### Постановка задачи 1.8

$$f(x)=10(x_1^4-5)^2+(x_2+2)^2-10 \quad \rightarrow \quad \mathrm{extr}$$

\begin{cases}
g_1(x)=x_1^2+5x_2^2-1 \le 0,\\
g_2(x)=-x_1 \le 0 \;(\Leftrightarrow x_1\ge 0),\\
g_3(x)=-x_2 \le 0 \;(\Leftrightarrow x_2\ge 0).
\end{cases}

Допустимое множество — компактный «квадрант эллипса» (четверть эллипса x_1^2+5x_2^2\le 1 в I квадранте).


## 1. Импорт и функции задачи


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize

def f(xy):
    x1, x2 = xy
    return 10.0 * (x1 ** 4 - 5.0) ** 2 + (x2 + 2.0) ** 2 - 10.0


def grad_f(xy):
    x1, x2 = xy
    df_dx1 = 80.0 * x1 ** 3 * (x1 ** 4 - 5.0)
    df_dx2 = 2.0 * (x2 + 2.0)
    return np.array([df_dx1, df_dx2], dtype=float)


constraints_scipy = [
    {"type": "ineq", "fun": lambda x: 1.0 - x[0] ** 2 - 5.0 * x[1] ** 2},
    {"type": "ineq", "fun": lambda x: x[0]},
    {"type": "ineq", "fun": lambda x: x[1]},
]

bounds = [(0.0, 1.0), (0.0, 1.0)]

np.set_printoptions(precision=12, suppress=True)
print("Готово: f(x), градиент, ограничения для SciPy")


## 2. График допустимого множества и линии уровня f(x)

Иллюстрация — графический результат выполнения программы.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

t = np.linspace(0, np.pi / 2, 400)
xe = np.cos(t)
ye = np.sin(t) / np.sqrt(5.0)
ax.fill(np.r_[xe, 0], np.r_[ye, 0], alpha=0.25, color="C0", label="Допустимое множество")

xg = np.linspace(0, 1, 260)
YG, XG = np.meshgrid(np.linspace(0, 0.55, 260), xg)
Z = np.full_like(XG, np.nan, dtype=float)
for i in range(XG.shape[0]):
    for j in range(XG.shape[1]):
        x1, x2 = XG[i, j], YG[i, j]
        if x1 ** 2 + 5 * x2 ** 2 <= 1.000001:
            Z[i, j] = f([x1, x2])

cs = ax.contour(XG, YG, Z, levels=18)
ax.clabel(cs, inline=True, fontsize=8)

ax.plot([1.0], [0.0], "r*", markersize=14, label="Минимум (численно)")
ax.plot([0.0], [1 / np.sqrt(5)], "ms", markersize=10, label="Максимум (численно)")

ax.set_xlabel("$x_1$")
ax.set_ylabel("$x_2$")
ax.set_title("Задание 1.8 (вариант 8): допустимое множество и уровни $f$")
ax.set_aspect("equal", adjustable="box")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 3. Аналитическое упрощение

$\displaystyle \frac{\partial f}{\partial x_2}=2(x_2+2)>0$ при $x_2\ge 0$, поэтому $f$ строго возрастает по $x_2$ на допустимом множестве ⇒ глобальный минимум при $x_2=0$.

При $x_2=0$ получаем $x_1\in[0,1]$. На этом отрезке ближайшее к 5 значение $x_1^4$ — при $x_1=1$, откуда минимум в точке $(1,0)$.

Глобальный максимум на компактном множестве сопоставляется по границе; численно максимум при $(0,\,1/\sqrt{5})$.


## 4. ККТ для минимума в точке кандидата $(1,0)$


In [ ]:
x_min = np.array([1.0, 0.0])
lam1, lam2, lam3 = 160.0, 0.0, 4.0


def grad_g1(xy):
    x1, x2 = xy
    return np.array([2 * x1, 10 * x2], dtype=float)


def grad_g2(_):
    return np.array([-1.0, 0.0], dtype=float)


def grad_g3(_):
    return np.array([0.0, -1.0], dtype=float)


G = grad_f(x_min) + lam1 * grad_g1(x_min) + lam2 * grad_g2(x_min) + lam3 * grad_g3(x_min)
print("Точка:", x_min)
print("Множители Лагранжа: λ1=%.6f λ2=%.6f λ3=%.6f" % (lam1, lam2, lam3))
print("‖∇x L‖:", np.linalg.norm(G))
g_vals = np.array([x_min[0] ** 2 + 5 * x_min[1] ** 2 - 1, -x_min[0], -x_min[1]])
print("Значения g1,g2,g3:", g_vals)


## 5. Численный поиск экстремумов (SciPy SLSQP, несколько стартов)


In [ ]:
starts = [
    [0.1, 0.05],
    [0.9, 0.05],
    [0.2, 0.35],
    [0.5, 0.1],
    [0.0, 0.3],
]

best = None
for s in starts:
    r = minimize(f, s, method="SLSQP", bounds=bounds, constraints=constraints_scipy)
    if best is None or r.fun < best.fun:
        best = r

print("Минимум: x =", best.x, "f =", best.fun)

best_m = None
for s in starts:
    r = minimize(lambda z: -f(z), s, method="SLSQP", bounds=bounds, constraints=constraints_scipy)
    if best_m is None or f(r.x) > f(best_m.x):
        best_m = r

print("Максимум: x =", best_m.x, "f =", f(best_m.x))

x_ray_max = np.array([0.0, 1.0 / np.sqrt(5.0)])
print("Контроль (луч x1=0, окружность активна):", x_ray_max, "f =", f(x_ray_max))


## 6. Условия второго порядка — два способа

**Способ A.** Одномерная редукция на $x_2=0$: $\varphi'(x_1)=80x_1^3(x_1^4-5)<0$ при $x_1\in(0,1)$ ⇒ минимум на правом конце $x_1=1$.

**Способ B.** Для вершины $(1,0)$ активны $g_1,g_3$. Численно проверяем знак квадратической формы $\Delta x^\top \nabla^2_{xx}L\,\Delta x$ для малого допустимого шага внутрь области.


In [ ]:
x1, x2 = 1.0, 0.0
h11 = 240 * x1**2 * (x1**4 - 5) + 640 * x1**6
h22 = 2.0
Hf = np.array([[h11, 0.0], [0.0, h22]])

lam1 = 160.0
H_L = Hf + lam1 * np.array([[2.0, 0.0], [0.0, 10.0]])


def feasible(x):
    return (
        x[0] >= -1e-9
        and x[1] >= -1e-9
        and x[0] ** 2 + 5 * x[1] ** 2 <= 1 + 1e-9
    )


def quad(d):
    return float(d @ H_L @ d)


x_star = np.array([1.0, 0.0])
d = np.array([-0.02, 0.01])
print("Δx:", d, "допустимо?", feasible(x_star + d), "Δxᵀ (∇²L) Δx =", quad(d))

d2 = np.array([-0.05, 0.0])
print("Δx:", d2, "допустимо?", feasible(x_star + d2), "Δxᵀ (∇²L) Δx =", quad(d2))

print("∇²f(1,0)=\n", Hf)
print("∇²_xx L=\n", H_L)


## 7. Сводная таблица точек


In [ ]:
import pandas as pd

rows = [
    {"точка": "(1, 0)", "f": f([1.0, 0.0]), "комментарий": "глобальный минимум / ККТ"},
    {"точка": "(0, 1/√5)", "f": f([0.0, 1 / np.sqrt(5.0)]), "комментарий": "глобальный максимум"},
    {"точка": "(0, 0)", "f": f([0.0, 0.0]), "комментарий": "вершина"},
]

display(pd.DataFrame(rows))


## 8. Выводы

- Выполнено задание **1.8** (**вариант 8**).
- ККТ для минимума выполняются в $(1,0)$ при $\lambda=(160,0,4)$.
- Значения: $f_{\min}=154$, $f_{\max}\approx 245{,}988854382$.
- Условия 2-го порядка проиллюстрированы редукцией на отрезке и численной проверкой формы $\nabla^2_{xx}L$.
